# 02 · Perfil dos dados · Olist
Análise exploratória que fundamenta as regras de negócio aplicadas na silver.
Cada consulta abaixo responde a uma decisão do projeto.

### 1. Peso de cada categoria
**Decisão que fundamenta:** o de-para de categorias (`de_para_categorias.csv`).

**Achado:** roupas e calçados somam só 611 itens (0,5%). O mix adotado inclui moda, acessórios, beleza, casa e esporte, totalizando 43.034 itens (38% do Olist).

In [0]:
-- Databricks SQL
SELECT p.product_category_name                                   AS categoria,
       COUNT(*)                                                  AS itens,
       ROUND(SUM(CAST(i.price AS DECIMAL(12,2))), 2)             AS receita,
       ROUND(100 * SUM(CAST(i.price AS DECIMAL(12,2)))
             / SUM(SUM(CAST(i.price AS DECIMAL(12,2)))) OVER (), 2) AS pct_receita
FROM horizonte.bronze.olist_order_items i
LEFT JOIN horizonte.bronze.olist_products p USING (product_id)
GROUP BY 1
ORDER BY receita DESC;

### 2. Status dos pedidos
**Decisão que fundamenta:** a regra de pedido válido.

**Achado:** 97% dos pedidos estão `delivered`. São excluídos `canceled` (625) e `unavailable` (609). Os demais status são vendas em andamento e entram, pois a venda é contada na data da compra.

In [0]:
SELECT order_status, COUNT(*) AS pedidos
FROM horizonte.bronze.olist_orders
GROUP BY 1
ORDER BY 2 DESC;

### 3. Período coberto
**Decisão que fundamenta:** o deslocamento de datas.

**Achado:** compras de 04/09/2016 a 17/10/2018. O deslocamento de 418 semanas (2.926 dias) alinha a Black Friday de 2017 com a de 2025 e preserva o dia da semana.

In [0]:
SELECT MIN(CAST(order_purchase_timestamp AS TIMESTAMP)) AS primeira_compra,
       MAX(CAST(order_purchase_timestamp AS TIMESTAMP)) AS ultima_compra
FROM horizonte.bronze.olist_orders;

### 4. Volume mensal
**Decisão que fundamenta:** a janela útil (`periodo_completo`).

**Achados:**
- Set–dez/2016 e set–out/2018 são quase vazios (349 pedidos no total); nov/2016 não tem nenhum pedido. Janela útil: jan/2017 a ago/2018.
- Nov/2017 tem 7.544 pedidos, +63% sobre outubro: efeito da Black Friday, a única presente nos dados.
- 2017 mostra crescimento da própria plataforma Olist (de 800 para cerca de 4.600 pedidos/mês), não sazonalidade. Ponto de atenção para o modelo de previsão.

In [0]:
SELECT date_trunc('MONTH', CAST(order_purchase_timestamp AS TIMESTAMP)) AS mes,
       COUNT(*) AS pedidos
FROM horizonte.bronze.olist_orders
GROUP BY 1 ORDER BY 1;

### 5. Pedidos sem itens
**Decisão que fundamenta:** a regra de pedido válido exige ao menos um item.

**Achado:** 775 pedidos não têm itens; 767 são cancelados ou indisponíveis. Os 8 restantes também são excluídos. Atenção: 461 pedidos cancelados **têm** itens, por isso a exclusão é feita pelo status, não só pelo join.

In [0]:
-- funciona igual nos dois motores, mudando só o nome das tabelas
SELECT o.order_status, COUNT(*) AS pedidos_sem_itens
FROM horizonte.bronze.olist_orders o
LEFT JOIN horizonte.bronze.olist_order_items i USING (order_id)
WHERE i.order_id IS NULL
GROUP BY 1;